<a href="https://colab.research.google.com/github/Mogi21/pytorch-deep-learning-labs/blob/main/05_pytorch_rnns_timeseries.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import Tuple, Dict, Optional

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# ==========================================
# 1. CONFIGURATION & DEVICE SETUP
# ==========================================
def set_seed(seed: int = 42) -> None:
    """Ensures deterministic behavior across random number generators."""
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Executing sequence pipeline on device: {device}")

# ==========================================
# 2. TIME-SERIES DATA GENERATION
# ==========================================
def generate_complex_series(batch_size: int, n_steps: int) -> np.ndarray:
    """
    Generates a synthetic dataset of superimposed sine waves with varying
    frequencies, phases, and noise to simulate complex dynamical systems.
    """
    f1, f2, f3 = np.random.rand(3, batch_size, 1)
    o1, o2, o3 = np.random.rand(3, batch_size, 1)
    t = np.linspace(0, 1, n_steps)

    # Superimpose slow, medium, and fast components
    s  = 0.3 * np.sin((t - o1) * (f1 *  4 +  4))
    s += 0.4 * np.sin((t - o2) * (f2 * 20 + 20))
    s += 0.2 * np.sin((t - o3) * (f3 * 40 + 40))
    s += 0.05 * (np.random.rand(batch_size, n_steps) - 0.5) # Add stochastic noise
    return s[..., np.newaxis].astype(np.float32)

# Generate sequences: First 200 steps predict the 201st step
SEQ_LEN = 200
series = generate_complex_series(8000, SEQ_LEN + 1)
X_train, y_train = series[:6000, :SEQ_LEN], series[:6000, -1]
X_valid, y_valid = series[6000:7500, :SEQ_LEN], series[6000:7500, -1]
X_test,  y_test  = series[7500:, :SEQ_LEN],  series[7500:, -1]

def to_tensor(x: np.ndarray) -> torch.Tensor:
    return torch.as_tensor(x, dtype=torch.float32, device=device)

# ==========================================
# 3. RECURRENT ARCHITECTURES
# ==========================================
class ScratchRNNCell(nn.Module):
    """
    A manual implementation of a vanilla Recurrent Neural Network cell.
    Demonstrates the underlying linear transformations and hidden state propagation
    without relying on PyTorch's fused backend.
    """
    def __init__(self, input_size: int, hidden_size: int):
        super().__init__()
        self.W_xh = nn.Linear(input_size, hidden_size)
        self.W_hh = nn.Linear(hidden_size, hidden_size, bias=False)

    def forward(self, x_t: torch.Tensor, h_prev: torch.Tensor) -> torch.Tensor:
        return torch.tanh(self.W_xh(x_t) + self.W_hh(h_prev))

class ManualRNNForecaster(nn.Module):
    """Integrates the ScratchRNNCell into a full unrolled sequence model."""
    def __init__(self, input_size: int = 1, hidden_size: int = 20):
        super().__init__()
        self.hidden_size = hidden_size
        self.cell = ScratchRNNCell(input_size, hidden_size)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch, seq_len, _ = x.shape
        h = torch.zeros(batch, self.hidden_size, device=x.device)
        for t in range(seq_len):
            h = self.cell(x[:, t, :], h)
        return self.fc(h)

class OptimizedForecaster(nn.Module):
    """
    A generalized sequence model utilizing PyTorch's optimized CUDNN backends.
    Supports standard RNN, LSTM (to combat vanishing gradients), and GRU architectures.
    """
    def __init__(self, rnn_type: str = 'LSTM', input_size: int = 1, hidden_size: int = 32, num_layers: int = 1):
        super().__init__()
        if rnn_type == 'RNN':
            self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)
        elif rnn_type == 'GRU':
            self.rnn = nn.GRU(input_size, hidden_size, num_layers, batch_first=True)
        else:
            self.rnn = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)

        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out, _ = self.rnn(x)
        # Extract the hidden state of the final timestep for the regression head
        return self.fc(out[:, -1, :])

class DeepSequenceClassifier(nn.Module):
    """
    A deep recurrent architecture designed for sequence classification.
    Features isolated recurrent blocks, Batch Normalization on temporal outputs,
    and Dropout for regularization.
    """
    def __init__(self, input_size: int = 1, hidden_size: int = 20, num_classes: int = 4, dropout_prob: float = 0.3):
        super().__init__()
        self.rnn1 = nn.RNN(input_size, hidden_size, num_layers=1, batch_first=True)
        self.rnn2 = nn.RNN(hidden_size, hidden_size, num_layers=1, batch_first=True)
        self.bn = nn.BatchNorm1d(hidden_size)
        self.dropout = nn.Dropout(dropout_prob)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out, _ = self.rnn1(x)
        out, _ = self.rnn2(out)

        # Route final timestep through normalization and regularization
        final_state = out[:, -1, :]
        norm_state = self.bn(final_state)
        drop_state = self.dropout(norm_state)
        return self.fc(drop_state)

# ==========================================
# 4. TRAINING ENGINE
# ==========================================
class SequenceTrainer:
    """Unified execution engine for sequence regression and classification."""
    def __init__(self, model: nn.Module, device: torch.device, task_type: str = 'regression'):
        self.model = model.to(device)
        self.device = device
        self.task_type = task_type
        self.criterion = nn.MSELoss() if task_type == 'regression' else nn.CrossEntropyLoss()

    def train(self, X_train: np.ndarray, y_train: np.ndarray,
              X_val: np.ndarray, y_val: np.ndarray,
              epochs: int = 10, lr: float = 1e-3, batch_size: int = 128, clip_grad: float = 1.0) -> Dict[str, list]:

        optimizer = optim.Adam(self.model.parameters(), lr=lr)
        history = {"train_loss": [], "val_loss": [], "val_metric": []}

        X_t = to_tensor(X_train)
        y_t = to_tensor(y_train) if self.task_type == 'regression' else torch.tensor(y_train, dtype=torch.long).to(self.device)
        if self.task_type == 'regression': y_t = y_t.view(-1, 1)

        loader = DataLoader(TensorDataset(X_t, y_t), batch_size=batch_size, shuffle=True)

        for ep in range(epochs):
            self.model.train()
            ep_loss = 0.0

            for xb, yb in loader:
                optimizer.zero_grad()
                pred = self.model(xb)
                loss = self.criterion(pred, yb)
                loss.backward()
                # Gradient clipping to prevent exploding gradients in RNNs
                if clip_grad:
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), clip_grad)
                optimizer.step()
                ep_loss += loss.item() * len(xb)

            train_loss = ep_loss / len(X_train)
            val_loss, val_metric = self.evaluate(X_val, y_val)

            history["train_loss"].append(train_loss)
            history["val_loss"].append(val_loss)
            history["val_metric"].append(val_metric)

            metric_name = "MSE" if self.task_type == 'regression' else "Acc"
            print(f"Epoch {ep+1:02d}/{epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val {metric_name}: {val_metric:.4f}")

        return history

    def evaluate(self, X: np.ndarray, y: np.ndarray) -> Tuple[float, float]:
        self.model.eval()
        X_t = to_tensor(X)
        y_t = to_tensor(y) if self.task_type == 'regression' else torch.tensor(y, dtype=torch.long).to(self.device)
        if self.task_type == 'regression': y_t = y_t.view(-1, 1)

        with torch.no_grad():
            pred = self.model(X_t)
            loss = self.criterion(pred, y_t).item()
            if self.task_type == 'regression':
                metric = loss # MSE
            else:
                metric = (pred.argmax(dim=1) == y_t).float().mean().item() # Accuracy
        return loss, metric

# ==========================================
# 5. EXPERIMENT 1: FORECASTING (LSTM vs GRU)
# ==========================================
print("--- Training Optimized LSTM Forecaster ---")
lstm_model = OptimizedForecaster(rnn_type='LSTM', hidden_size=32)
lstm_trainer = SequenceTrainer(lstm_model, device, task_type='regression')
lstm_trainer.train(X_train, y_train, X_valid, y_valid, epochs=10, lr=1e-3)

lstm_test_mse, _ = lstm_trainer.evaluate(X_test, y_test)
print(f"LSTM Test MSE: {lstm_test_mse:.5f}\n")

# ==========================================
# 6. EXPERIMENT 2: SEQUENCE CLASSIFICATION
# ==========================================
print("--- Training Deep Sequence Classifier on Real Data ---")
if os.path.exists('train.csv') and os.path.exists('test_inputs.csv'):
    # Load standardized time-series datasets
    train_df = pd.read_csv('train.csv')
    test_df = pd.read_csv('test_inputs.csv')

    # Extract temporal sequences and target classes
    seq_cols = [c for c in train_df.columns if c.startswith('t')]
    X_cls_raw = train_df[seq_cols].values[..., np.newaxis].astype(np.float32)
    y_cls_raw = train_df['label'].values

    # Walk-forward validation split
    split_idx = int(len(X_cls_raw) * 0.8)
    X_c_train, X_c_val = X_cls_raw[:split_idx], X_cls_raw[split_idx:]
    y_c_train, y_c_val = y_cls_raw[:split_idx], y_cls_raw[split_idx:]

    # Train deep classification architecture
    classifier = DeepSequenceClassifier(input_size=1, hidden_size=32, num_classes=4)
    cls_trainer = SequenceTrainer(classifier, device, task_type='classification')
    cls_trainer.train(X_c_train, y_c_train, X_c_val, y_c_val, epochs=15, lr=2e-3)

    # Inference on unlabelled test data
    classifier.eval()
    X_c_test = test_df[seq_cols].values[..., np.newaxis].astype(np.float32)
    with torch.no_grad():
        test_logits = classifier(to_tensor(X_c_test))
        test_preds = test_logits.argmax(dim=1).cpu().numpy()

    pd.DataFrame(test_preds, columns=['prediction']).to_csv('submission_4.csv', index=False)
    print("Classification inference complete. Saved to 'submission_4.csv'.")
else:
    print("[Notice] 'train.csv' or 'test_inputs.csv' not found. Skipping final classification step.")

Executing sequence pipeline on device: cpu
--- Training Optimized LSTM Forecaster ---
Epoch 01/10 | Train Loss: 0.1062 | Val Loss: 0.0629 | Val MSE: 0.0629
Epoch 02/10 | Train Loss: 0.0336 | Val Loss: 0.0116 | Val MSE: 0.0116
Epoch 03/10 | Train Loss: 0.0054 | Val Loss: 0.0030 | Val MSE: 0.0030
Epoch 04/10 | Train Loss: 0.0025 | Val Loss: 0.0025 | Val MSE: 0.0025
Epoch 05/10 | Train Loss: 0.0022 | Val Loss: 0.0021 | Val MSE: 0.0021
Epoch 06/10 | Train Loss: 0.0019 | Val Loss: 0.0019 | Val MSE: 0.0019
Epoch 07/10 | Train Loss: 0.0019 | Val Loss: 0.0018 | Val MSE: 0.0018
Epoch 08/10 | Train Loss: 0.0017 | Val Loss: 0.0016 | Val MSE: 0.0016
Epoch 09/10 | Train Loss: 0.0015 | Val Loss: 0.0017 | Val MSE: 0.0017
Epoch 10/10 | Train Loss: 0.0015 | Val Loss: 0.0015 | Val MSE: 0.0015
LSTM Test MSE: 0.00142

--- Training Deep Sequence Classifier on Real Data ---
[Notice] 'train.csv' or 'test_inputs.csv' not found. Skipping final classification step.
